In [1]:
# Importing libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
import category_encoders as ce
import joblib
from sklearn.cluster import KMeans
from geopy.distance import geodesic
from feature_engine.outliers import Winsorizer

# Load the raw dataset
df = pd.read_csv('../../data/raw/Fraudtrain.csv')

# Convert trans_date_trans_time to pandas datetime
df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])
df['trans_hour'] = df['trans_date_trans_time'].dt.hour
df['trans_month'] = df['trans_date_trans_time'].dt.month
df['trans_dayofweek'] = df['trans_date_trans_time'].dt.day_name()

# Convert unix_time to datetime
df['transaction_time'] = pd.to_datetime(df['unix_time'], unit='s')
df = df.sort_values(by=['cc_num', 'transaction_time'])

# Calculate the previous transaction time using shift
df['unix_time_prev_trans'] = df.groupby(by=['cc_num'])['unix_time'].shift(1)
df = df.assign(unix_time_prev_trans=df['unix_time_prev_trans'].fillna(df['unix_time'] - 86400))
df['timedelta_last_trans'] = (df['unix_time'] - df['unix_time_prev_trans']) // 60

# Convert dob to datetime and calculate customer age
df['dob'] = pd.to_datetime(df['dob'])
df['cust_age'] = (df['trans_date_trans_time'] - df['dob']).dt.days // 365

# Define the Haversine function
def haversine(lat1, lon1, lat2, lon2):
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lon2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    km = 6371 * c
    return km

# Calculate distance from previous merchant
df['prev_merch_lat'] = df.groupby('cc_num')['merch_lat'].shift(1)
df['prev_merch_long'] = df.groupby('cc_num')['merch_long'].shift(1)
df = df.assign(prev_merch_lat=df['prev_merch_lat'].fillna(df['merch_lat']),
               prev_merch_long=df['prev_merch_long'].fillna(df['merch_long']))
df['distance_from_prev_merch'] = df.apply(lambda row: haversine(row['merch_lat'], row['merch_long'], row['prev_merch_lat'], row['prev_merch_long']), axis=1)

# Function to check if the transaction is during business hours
def is_business_hours(hour):
    return 1 if 9 <= hour <= 18 else 0

# Function to check if the transaction occurred on a weekend
def is_weekend(day):
    return 1 if day in ['Saturday', 'Sunday'] else 0

# Add business hours and weekend features
df['business_hours'] = df['trans_hour'].apply(is_business_hours)
df['weekend'] = df['trans_dayofweek'].apply(is_weekend)

# Add merchant location clusters
coords = df[['merch_lat', 'merch_long']].dropna().values
kmeans = KMeans(n_clusters=10, random_state=42)
df['merchant_cluster'] = kmeans.fit_predict(coords)

# Calculate the distance between customer city and merchant city
def calculate_customer_to_merchant_distance(row):
    customer_coords = (row['lat'], row['long'])
    merchant_coords = (row['merch_lat'], row['merch_long'])
    return geodesic(customer_coords, merchant_coords).kilometers

df['customer_to_merchant_distance'] = df.apply(calculate_customer_to_merchant_distance, axis=1)

# Calculate average transaction amount per credit card
df['avg_amt_per_cc'] = df.groupby('cc_num')['amt'].transform('mean')

# Calculate transaction frequency per day for the same credit card
df['trans_freq_per_day'] = df.groupby(['cc_num', df['transaction_time'].dt.date])['trans_date_trans_time'].transform('count')

# Create interaction features
df['amt_trans_hour_interaction'] = df['amt'] * df['trans_hour']
df['cust_age_amt_interaction'] = df['cust_age'] * df['amt']

# List of columns to drop
drop_cols = [
    'Unnamed: 0', 'street', 'zip', 'first', 'last', 'trans_num', 'trans_date_trans_time',
    'transaction_time', 'cc_num', 'unix_time', 'unix_time_prev_trans', 'lat', 'long',
    'merch_lat', 'merch_long', 'dob'
]

# Dropping the columns from the DataFrame
df = df.drop(columns=drop_cols)

# Split the dataset into training and validation sets
X = df.drop(columns=["is_fraud"])
y = df["is_fraud"]

X_train, X_val, y_train, y_val = train_test_split(X, y, stratify=y, test_size=0.3, random_state=42)



# Define the outlier capping function
def cap_outliers(df, columns, fold=1.5):
    capper = Winsorizer(capping_method='iqr', tail='both', fold=fold, variables=columns)
    capper.fit(df)
    df = capper.transform(df)
    return df, capper

# Columns with outliers to be capped
outlier_columns = ['amt', 'city_pop', 'timedelta_last_trans', 'distance_from_prev_merch', 'cust_age']


# Apply outlier capping to training and validation data
X_train_capped, capper = cap_outliers(X_train, outlier_columns)
X_val_capped = capper.transform(X_val)

# Apply WOE encoding and save encoders
def apply_woe_and_save_encoders(X_train, X_val, y_train, columns, encoder_filename):
    woe_encoders = []
    for col in columns:
        woe = ce.WOEEncoder(cols=[col])
        X_train_transformed = woe.fit_transform(X_train[[col]], y_train)
        X_val_transformed = woe.transform(X_val[[col]])
        new_col_name = f"{col}_WOE"
        X_train[new_col_name] = X_train_transformed[col]
        X_val[new_col_name] = X_val_transformed[col]
        woe_encoders.append(woe)
    joblib.dump(woe_encoders, encoder_filename)
    return X_train, X_val

# Categorical columns to encode
columns_to_encode = ['state', 'job', 'category', 'trans_dayofweek', 'merchant', 'city']

# Apply WOE encoding to training and validation data and save encoders
encoder_filename = 'woe_encoders_newFeatures.joblib'
X_train_encoded, X_val_encoded = apply_woe_and_save_encoders(X_train_capped, X_val_capped, y_train, columns_to_encode, encoder_filename)

# Apply one-hot encoding to the gender column
def apply_one_hot_encoding(X_train, X_val, column):
    ohe = OneHotEncoder(sparse_output=False, drop='first')
    ohe.fit(X_train[[column]])
    X_train_ohe = ohe.transform(X_train[[column]])
    X_val_ohe = ohe.transform(X_val[[column]])
    ohe_columns = ohe.get_feature_names_out([column])
    X_train_ohe_df = pd.DataFrame(X_train_ohe, columns=ohe_columns, index=X_train.index)
    X_val_ohe_df = pd.DataFrame(X_val_ohe, columns=ohe_columns, index=X_val.index)
    X_train = pd.concat([X_train, X_train_ohe_df], axis=1)
    X_val = pd.concat([X_val, X_val_ohe_df], axis=1)
    return X_train, X_val

# Apply one-hot encoding to the gender column in the WOE-encoded DataFrames
ohe_column = 'gender'
X_train_encoded, X_val_encoded = apply_one_hot_encoding(X_train_encoded, X_val_encoded, ohe_column)

# Drop the specified columns from the training and validation sets
columns_to_drop = ['category', 'gender', 'state', 'job', 'trans_dayofweek', 'city', 'merchant']
X_train_encoded = X_train_encoded.drop(columns=columns_to_drop)
X_val_encoded = X_val_encoded.drop(columns=columns_to_drop)

# Scale the features
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train_encoded)
X_val_scaled = scaler.transform(X_val_encoded)
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X_train_encoded.columns, index=X_train_encoded.index)
X_val_scaled_df = pd.DataFrame(X_val_scaled, columns=X_val_encoded.columns, index=X_val_encoded.index)


In [3]:
# Load the unseen dataset
df = pd.read_csv('../../data/raw/Fraudtest.csv')

# Apply the same preprocessing steps to the unseen data
df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])
df['trans_hour'] = df['trans_date_trans_time'].dt.hour
df['trans_month'] = df['trans_date_trans_time'].dt.month
df['trans_dayofweek'] = df['trans_date_trans_time'].dt.day_name()
df['transaction_time'] = pd.to_datetime(df['unix_time'], unit='s')
df = df.sort_values(by=['cc_num', 'transaction_time'])
df['unix_time_prev_trans'] = df.groupby(by=['cc_num'])['unix_time'].shift(1)
df = df.assign(unix_time_prev_trans=df['unix_time_prev_trans'].fillna(df['unix_time'] - 86400))
df['timedelta_last_trans'] = (df['unix_time'] - df['unix_time_prev_trans']) // 60
df['dob'] = pd.to_datetime(df['dob'])
df['cust_age'] = (df['trans_date_trans_time'] - df['dob']).dt.days // 365
df['prev_merch_lat'] = df.groupby('cc_num')['merch_lat'].shift(1)
df['prev_merch_long'] = df.groupby('cc_num')['merch_long'].shift(1)
df = df.assign(prev_merch_lat=df['prev_merch_lat'].fillna(df['merch_lat']),
               prev_merch_long=df['prev_merch_long'].fillna(df['merch_long']))
df['distance_from_prev_merch'] = df.apply(lambda row: haversine(row['merch_lat'], row['merch_long'], row['prev_merch_lat'], row['prev_merch_long']), axis=1)
df['business_hours'] = df['trans_hour'].apply(is_business_hours)
df['weekend'] = df['trans_dayofweek'].apply(is_weekend)
df['merchant_cluster'] = kmeans.predict(df[['merch_lat', 'merch_long']])
df['customer_to_merchant_distance'] = df.apply(calculate_customer_to_merchant_distance, axis=1)
df['avg_amt_per_cc'] = df.groupby('cc_num')['amt'].transform('mean')
df['trans_freq_per_day'] = df.groupby(['cc_num', df['transaction_time'].dt.date])['trans_date_trans_time'].transform('count')
df['amt_trans_hour_interaction'] = df['amt'] * df['trans_hour']
df['cust_age_amt_interaction'] = df['cust_age'] * df['amt']

# List of columns to drop
drop_cols = [
    'Unnamed: 0', 'street', 'zip', 'first', 'last', 'trans_num', 'trans_date_trans_time',
    'transaction_time', 'cc_num', 'unix_time', 'unix_time_prev_trans', 'lat', 'long',
    'merch_lat', 'merch_long', 'dob'
]

# Dropping the columns from the DataFrame
df = df.drop(columns=drop_cols)

# Separate features and target variable
X_unseen = df.drop(columns=["is_fraud"])
y_unseen = df["is_fraud"]

# Apply outlier capping to unseen data
X_unseen_capped = capper.transform(X_unseen)

# Load the saved WOE encoders
woe_encoders = joblib.load('woe_encoders_newFeatures.joblib')

# Apply WOE encoding
def apply_woe(X, woe_columns, woe_encoders):
    for col, encoder in zip(woe_columns, woe_encoders):
        X_transformed = encoder.transform(X[[col]])
        new_col_name = f"{col}_WOE"
        X[new_col_name] = X_transformed[col]
    return X

woe_columns = ['state', 'job', 'category', 'trans_dayofweek', 'merchant', 'city']
X_unseen_encoded = apply_woe(X_unseen_capped, woe_columns, woe_encoders)

# Apply one-hot encoding to the gender column
def apply_one_hot_encoding_unseen(X, column, ohe):
    X_ohe = ohe.transform(X[[column]])
    ohe_columns = ohe.get_feature_names_out([column])
    X_ohe_df = pd.DataFrame(X_ohe, columns=ohe_columns, index=X.index)
    X = pd.concat([X, X_ohe_df], axis=1)
    return X

X_unseen_encoded = apply_one_hot_encoding_unseen(X_unseen_encoded, 'gender', ohe)

# Drop the specified columns from the unseen set
X_unseen_encoded = X_unseen_encoded.drop(columns=columns_to_drop)

# Scale the features
X_unseen_scaled = scaler.transform(X_unseen_encoded)
X_unseen_scaled_df = pd.DataFrame(X_unseen_scaled, columns=X_unseen_encoded.columns, index=X_unseen_encoded.index)

# Display a sample of the final DataFrame for unseen data
print("\nFinal Unseen DataFrame Sample:")
print(X_unseen_scaled_df.head())


C:\Users\Black\anaconda3\Lib\site-packages\sklearn\base.py:486: UserWarning: X has feature names, but KMeans was fitted without feature names
  warnings.warn(


NameError: name 'ohe' is not defined

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, average_precision_score, roc_curve, precision_recall_curve, auc
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from collections import Counter
import numpy as np

# Calculate scale_pos_weight using the square root of the ratio
counter = Counter(y_train)
scale_pos_weight = np.sqrt(counter[0] / counter[1])

print("Scale Pos Weight:", scale_pos_weight)
print('Estimate for scale_pos_weight: %.3f' % scale_pos_weight)

# Define the initial XGBoost model
xgb_model = XGBClassifier(scale_pos_weight=scale_pos_weight, random_state=42)

# Train the model on the training data
xgb_model.fit(X_train_scaled_df, y_train)

# Make predictions on the validation data
y_val_pred = xgb_model.predict(X_val_scaled_df)
y_val_pred_proba = xgb_model.predict_proba(X_val_scaled_df)[:, 1]

# Evaluate the model on the validation data
print("Validation Accuracy:", accuracy_score(y_val, y_val_pred))
print("Validation Classification Report:\n", classification_report(y_val, y_val_pred))
print("Validation Confusion Matrix:\n", confusion_matrix(y_val, y_val_pred))
print("Validation ROC AUC Score:", roc_auc_score(y_val, y_val_pred_proba))
print("Validation PR AUC Score:", average_precision_score(y_val, y_val_pred_proba))

# Plot ROC Curve
fpr, tpr, _ = roc_curve(y_val, y_val_pred_proba)
roc_auc = auc(fpr, tpr)
plt.figure()
plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic')
plt.legend(loc="lower right")
plt.show()

# Plot Precision-Recall Curve
precision, recall, _ = precision_recall_curve(y_val, y_val_pred_proba)
pr_auc = auc(recall, precision)
plt.figure()
plt.plot(recall, precision, color='b', lw=2, label='PR curve (area = %0.2f)' % pr_auc)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend(loc="lower left")
plt.show()

In [ ]:
# Make predictions on the unseen data with the base model
y_unseen_pred = xgb_model.predict(X_unseen_scaled_df)
y_unseen_pred_proba = xgb_model.predict_proba(X_unseen_scaled_df)[:, 1]

# Evaluate the model on the unseen data
print("Unseen Data Accuracy:", accuracy_score(y_unseen, y_unseen_pred))
print("Unseen Data Classification Report:\n", classification_report(y_unseen, y_unseen_pred))
print("Unseen Data Confusion Matrix:\n", confusion_matrix(y_unseen, y_unseen_pred))
print("Unseen Data ROC AUC Score:", roc_auc_score(y_unseen, y_unseen_pred_proba))
print("Unseen Data PR AUC Score:", average_precision_score(y_unseen, y_unseen_pred_proba))

# Plot ROC Curve
fpr, tpr, _ = roc_curve(y_unseen, y_unseen_pred_proba)
roc_auc = auc(fpr, tpr)
plt.figure()
plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic')
plt.legend(loc="lower right")
plt.show()

# Plot Precision-Recall Curve
precision, recall, _ = precision_recall_curve(y_unseen, y_unseen_pred_proba)
pr_auc = auc(recall, precision)
plt.figure()
plt.plot(recall, precision, color='b', lw=2, label='PR curve (area = %0.2f)' % pr_auc)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend(loc="lower left")
plt.show()